The Front-End UX: Markdown Rendering and Real-Time Streaming

In AI Engineering, TTFT (Time To First Token) is more important than total generation time. If a user sees the AI "typing" in real-time, they are willing to wait much longer for the final answer.


Today, we move from "Buffer-and-Blast" (waiting for the full response) to Streaming Architecture. We will use Python Generators to pipe tokens into Gradio as they arrive, rendering beautiful Markdown code blocks and formatting on the fly.

In [ ]:
import gradio as gr
from litellm import completion
from dotenv import load_dotenv
import time

load_dotenv(override=True)

print("🌊 Streaming UX Environment Initialized!")

1. The Generator Pattern: Piping Tokens

Instead of a standard function that returns a string, we write a Generator that yields strings. Gradio's UI components are designed to listen for these yields and update the interface instantly.

In [ ]:
def stream_ai_response(prompt, model_choice="openai/gpt-4o-mini"):
    """
    A generator function that yields tokens as they arrive.
    """
    # 1. Start the streaming request
    response = completion(
        model=model_choice, 
        messages=[{"role": "user", "content": prompt}],
        stream=True # CRITICAL: This tells the API to use SSE
    )
    
    partial_text = ""
    for chunk in response:
        # 2. Extract the token content
        token = chunk.choices[0].delta.content
        if token:
            partial_text += token
            # 3. YIELD the accumulated text to the UI
            yield partial_text

2. Wiring the Streaming UI

We will use gr.Blocks to create a professional layout that emphasizes Markdown rendering. Notice how the outputs parameter of our button click is tied directly to our generator.

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as stream_demo:
    gr.Markdown("# ⚡ High-Performance Streaming Architect")
    gr.Markdown("Watch how the AI renders complex Markdown and Code Blocks in real-time.")
    
    with gr.Row():
        user_input = gr.Textbox(
            label="Technical Prompt", 
            placeholder="E.g., Write a Python function for a Binary Search Tree with comments.",
            lines=3
        )
    
    submit_btn = gr.Button("Generate with Stream", variant="primary")
    
    # gr.Markdown components are highly optimized for streaming content
    output_display = gr.Markdown(label="Real-Time Output")

    # WIRE THE EVENT
    submit_btn.click(
        fn=stream_ai_response, 
        inputs=user_input, 
        outputs=output_display
    )

print("🚀 Streaming UI Wired and Ready.")
stream_demo.launch()